# DGMAE Figure 10 — discrepancy weight
Sweep λ from 0 to 1 in increments of 0.1 for Roman, Actor, Citeseer, Photo and Cora. The runner follows Eq. 13 with one-hop discrepancy and selects the linear probe on validation accuracy. Cora uses its 140/500/1,000 public split. Roman uses new seeded 48/32/20 splits. Each full point uses ten seeds. Roman can be divided into separate batches by changing DATASETS and WEIGHTS.

In [ ]:
MODE = "configure"  # "configure", "smoke" or "full"
DATASETS = ["actor", "citeseer", "photo", "cora"]
WEIGHTS = [i / 10 for i in range(11)]
SEEDS = list(range(10))
RESUME = False  # Restore the attached partial Figure 10 v1 output.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

assert MODE in {"configure", "smoke", "full"}, MODE

workdir = Path("/kaggle/working")
# Kaggle saves the results directory as output. Keep setup files in session-local /tmp.
setup_dir = Path("/tmp/dgmae-figure10")
repo = setup_dir / "cs313"
upstream = setup_dir / "upstream"
experiment_dir = repo / "experiments" / "ablation"
experiment_commit = "e0d34a19035c7a69b97265498aedef31c4e9a136"
upstream_commit = "0e770864b5283b01e2597d6f092e8a812aa5031a"

def checkout(url, commit, destination):
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", url, commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()
    if actual != commit:
        raise RuntimeError(f"Expected commit {commit}, found {actual}")

if MODE == "configure":
    print("Set MODE to smoke or full to enable execution.")
else:
    subprocess.run(["nvidia-smi", "-L"], check=True)
    checkout("https://github.com/phud774/CS313.R11.git", experiment_commit, repo)
    checkout("https://github.com/zhengziyu77/DGMAE.git", upstream_commit, upstream)
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
    subprocess.run(["bash", str(experiment_dir / "setup_kaggle.sh"), str(setup_dir)], check=True)

In [ ]:
if MODE != "configure":
    if RESUME and MODE != "full":
        raise ValueError("Resume requires full mode.")
    launcher = experiment_dir / "run_kaggle_python.sh"
    output = workdir / "dgmae-figure10"
    if RESUME:
        import json

        source_candidates = [
            path for path in Path("/kaggle/input").rglob("dgmae-figure10")
            if (path / "paper" / "full").is_dir()
        ]
        if len(source_candidates) != 1:
            raise RuntimeError(f"Expected the Figure 10 v1 output attachment, found {source_candidates}")
        prior = source_candidates[0] / "paper" / "full"
        expected = {
            (f"lambda-{weight:g}", dataset, seed)
            for weight in WEIGHTS if weight <= 0.5
            for dataset in DATASETS
            for seed in SEEDS
        }
        expected.update(("lambda-0.6", "actor", seed) for seed in SEEDS)
        expected.update(("lambda-0.6", "citeseer", seed) for seed in range(7))
        seed_files = list(prior.glob("lambda-*/*/dataset-default/full/seed-*.json"))
        actual = {(p.parts[-5], p.parts[-4], int(p.stem[5:])) for p in seed_files}
        if actual != expected or len(actual) != 257:
            raise RuntimeError(f"Expected the 257 full seed results from v1, found {len(actual)}")
        for run_dir in {p.parent for p in seed_files}:
            settings = json.loads((run_dir / "settings.json").read_text())
            if (settings["experiment_commit"], settings["upstream_commit"], settings["smoke"]) != (
                experiment_commit, upstream_commit, False,
            ):
                raise RuntimeError(f"Unexpected settings in resumed run {run_dir}")
        shutil.copytree(source_candidates[0], output, dirs_exist_ok=True)
        print(f"Restored {len(actual)} completed seeds from Figure 10 v1.", flush=True)

    env = dict(os.environ, DGL_DOWNLOAD_DIR=str(setup_dir / "datasets"), PYTHONUNBUFFERED="1")
    base = [
        "bash", str(launcher), str(setup_dir), "-B", str(experiment_dir / "run_ablation.py"),
        "--upstream", str(upstream), "--output", str(output), "--protocol", "paper",
        "--device", "cuda:0", "--dataset", *DATASETS, "--variant", "full",
        "--discrepancy-neighborhood", "one-hop", "--roman-split", "paper-ratio",
    ]
    for weight in (() if RESUME else (0, 1)):
        subprocess.run(base + ["--smoke", "--seeds", "0", "--discrepancy-weight", str(weight)],
                       env=env, check=True)
    if MODE == "full":
        for weight in (w for w in WEIGHTS if not RESUME or w >= 0.6):
            subprocess.run(base + ["--seeds", *map(str, SEEDS), "--discrepancy-weight", str(weight)],
                           env=env, check=True)

In [ ]:
if MODE != "configure":
    import json
    import matplotlib.pyplot as plt
    import pandas as pd

    rows = []
    for path in sorted((output / "paper" / MODE).glob("lambda-*/*/*/full/summary.json")):
        row = json.loads(path.read_text())
        row["lambda"] = float(path.parents[3].name.removeprefix("lambda-"))
        rows.append(row)
    table = pd.DataFrame(rows).sort_values(["dataset", "lambda"])
    if MODE == "full":
        assert len(table) == len(DATASETS) * len(WEIGHTS)
        assert table["runs"].eq(len(SEEDS)).all()
    display(table)
    table.to_csv(output / f"figure10_{MODE}.csv", index=False)

    if MODE == "full":
        fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
        for dataset, group in table.groupby("dataset"):
            ax.plot(group["lambda"], group["mean_accuracy_pct"], marker="o", label=dataset.capitalize())
        ax.set(xlabel="Discrepancy weight λ", ylabel="Mean test accuracy (%)", xlim=(0, 1))
        ax.legend()
        ax.grid(alpha=0.25)
        for extension in ("png", "pdf"):
            fig.savefig(output / f"figure10.{extension}", dpi=180)
        display(fig)
        plt.close(fig)